In [ ]:
# Authentication and authorization
from google.colab import auth
auth.authenticate_user()

from google.auth import default
creds, _ = default()

# URL: https://docs.gspread.org/en/v5.7.0/
import gspread
gc = gspread.authorize(creds)

from google.colab import drive
drive.mount('/content/drive')

# Constants
PROJECT_FOLDER = '/content/drive/MyDrive/Colab Notebooks/DataScientest/feb25_bds_streamlit/'

### Data
We have data consisting of 25,000 film reviews. The objective is to train the word embeddings matrices on this data.

(a) Load the data file under the name df and explore it.
Since the Word2Vec approach only requires text, we do not need the "sentiment" column of the dataframe.

(b) Delete the "sentiment" column from df.

In [ ]:
import pandas as pd

df = pd.read_csv("MovieReview.csv")

display(df.head())
print(df.shape)

df = df.drop(columns=["sentiment"])

(c) Add the following code to clean up the data and to remove stopwords.

In [ ]:
import re
import unicodedata
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt_tab')
stop_words = stopwords.words('english')

# Converts the unicode file to ascii
def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn')

def preprocess_sentence(w):
    w = unicode_to_ascii(w.lower().strip())
    # creating a space between a word and the punctuation following it
    # eg: "he is a boy." => "he is a boy ."
    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    w = re.sub(r'[" "]+', " ", w)
    # replacing everything with space except (a-z, A-Z, ".", "?", "!", ",")
    w = re.sub(r"[^a-zA-Z?.!]+", " ", w)
    w = re.sub(r'\b\w{0,2}\b', '', w)

    # remove stopword
    mots = word_tokenize(w.strip())
    mots = [mot for mot in mots if mot not in stop_words]
    return ' '.join(mots).strip()

df.review = df.review.apply(lambda x: preprocess_sentence(x))
df.head()

### Tokens

The Tokenizer class of tensorflow.keras.preprocessing.text allows to vectorize a corpus of text. Indeed, it transforms each text into a sequence of integers, each integer being the index of a token in a dictionary. The num_words argument limits the size of the dictionary.

The fit_on_texts method updates the dictionary from a list of texts.

(d) Define a tokenizer object using the tensorflow.keras.preprocessing.text tokenizer constructor, specifying a dictionary word limit of 10000.
(e) Update the tokenizer dictionary using the fit_on_texts method.

In [ ]:
import tensorflow as tf
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=10000)
tokenizer.fit_on_texts(df.review)

(e) Store the word-index matching dictionary in the word2idx variable, and the index-word matching dictionary in the idx2word variable, using the word_index attribute of the tokenizer.

(f) Store the size of the dictionary in the vocab_size variable, using the num_words attribute of the tokenizer.

In [ ]:
word2idx = tokenizer.word_index
idx2word = tokenizer.index_word
vocab_size = tokenizer.num_words

### Modelling

We will implement a particular Word2Vec model : the Continuous Bag Of Words (CBOW) model. The CBOW model tries to predict a word thanks to its context, ie thanks to the words close to it in the text. The inputs of the model are the context words and the output of the model is a prediction probability of the target word.

More precisely, the CBOW model is a neural network with 3 layers: an input layer, a hidden layer and an output layer. The hidden layer consists of an Embedding layer which transforms each input word into an embedding vector, so that the embedding matrix is learned as the training progresses. There is also a pooling layer (GlobalAveragePooling1D) which sums up the different embeddings to get a good dimensional result. Finally, the prediction of the target word is done thanks to a Dense layer.

(g) Add the following code to create the data set (X, Y).

In [ ]:
import numpy as np

def sentenceToData(tokens, WINDOW_SIZE):
    window = np.concatenate((np.arange(-WINDOW_SIZE,0),np.arange(1,WINDOW_SIZE+1)))
    X,Y=([],[])
    for word_index, word in enumerate(tokens) :
        if ((word_index - WINDOW_SIZE >= 0) and (word_index + WINDOW_SIZE <= len(tokens) - 1)) :
            X.append(word)
            Y.append([tokens[word_index-i] for i in window])
    return X, Y


WINDOW_SIZE = 5

X, Y = ([], [])
for review in df.review:
    for sentence in review.split("."):
        word_list = tokenizer.texts_to_sequences([sentence])[0]
        if len(word_list) >= WINDOW_SIZE:
            Y1, X1 = sentenceToData(word_list, WINDOW_SIZE//2)
            X.extend(X1)
            Y.extend(Y1)
    
X = np.array(X).astype(int)
y = np.array(Y).astype(int).reshape([-1,1])

(h) Create the model architecture. The Embedding layer will take an input of size 10000 and an output of size 300. The Dense layer will consist of 10000 neurons and a SoftMax activation function.

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D

embedding_dim = 300

model = Sequential()
model.add(Embedding(vocab_size, embedding_dim))
model.add(GlobalAveragePooling1D())
model.add(Dense(vocab_size, activation='softmax'))

(i) Compile the model.
(j) Train the model during 50 epochs.
Note : The model is complexe and the training can take several hours.

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.fit(X, y, batch_size = 128, epochs=1)

### Saving the model
As explained in the third notebook, it is important to save the model such that it is not re-trained each time Streamlit is deployed. It is even more important in Deep Learning, where the models are complex and where training them takes time.

With Keras, it is possible to save a trained model in its entirety (architecture, weights/filters learned during training and compilation information). This is done using the H5 format.

(g) Save the model in H5 format using the save method in Keras.

In [ ]:
model.save("word2vec.h5") 

### Creating the Python file for Streamlit
Now that the Deep Learning model has been trained and saved, we can move on to Streamlit. As usual, we use a Python code editor (e.g. VSCode or Spyder) to obtain a .py file.

(h) Create a .py file which will contain the Python script dedicated to the Streamlit application. Save it in the same folder as the other project files.

(i) Give a title to Streamlit.

(j) Load the weights of the registered model using the load_weights method in Keras.

In [ ]:
# see movie_review.py